# analyze 분류 모델 — 학습 · 분류 원리 · 벡터 유사도

이 노트북은 실시간 필터(`impl.py` 의 `run()`)에 넣는 분류 모델을 **직접 돌려 보는 용도**입니다. 앱·테스트는 이 파일을 쓰지 않습니다.

1. **학습**: 학습률 · 배치 · epoch 가 실제로 어떻게 돌아가는지 로그로 봅니다.
2. **분류 원리**: 인코더 모델이 문장을 벡터로 바꾸고, 그 벡터가 어떻게 유형 5개의 점수가 되는지 한 단계씩 봅니다.
3. **벡터 유사도**: 유형별 대표 문장들과의 코사인 유사도를 재고, 실제 분류 층과 비교합니다.

실행 방법 (주피터는 프로젝트 의존성이 아니라서 일회용으로 띄웁니다):
```bash
uv sync --extra analyze
uv run --with jupyterlab jupyter lab src/clean_chat/modules/analyze/notebooks/model_demo.ipynb
```
- 시험 문장(`split=test`)은 이 노트북 어디에서도 쓰지 않습니다. 학습·대표 문장은 모두 `train` 행입니다.
- 문장은 전부 AI 가 쓴 가상 문장(검수 전)입니다.

## 0. 설정

여기 숫자를 바꾸고 전체를 다시 실행하면 됩니다.

| 이름 | 뜻 |
|---|---|
| `BASE` | 출발점이 되는 사전학습 모델 (Hugging Face 이름) |
| `LR` | 학습률 — 한 번에 가중치를 얼마나 크게 고칠지 |
| `BATCH_SIZE` | 배치 크기 — 한 번에 몇 문장을 묶어 보고 가중치를 고칠지 |
| `EPOCHS` | 학습 문장 전체를 최대 몇 바퀴 볼지 |
| `PATIENCE` | 검증 점수가 이만큼의 epoch 동안 안 오르면 일찍 멈춤 |
| `VAL_RATIO` | 학습 문장 중 검증용으로 떼어 둘 비율 (학습에는 안 씀) |

In [1]:
import json
import os
import random
import time
import warnings

import torch
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging

from clean_chat.core.config import resolve_repo_path
from clean_chat.modules.analyze import evaluate, impl, model, train

warnings.filterwarnings("ignore")
hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()

BASE = "beomi/KcELECTRA-base"  # 소형으로 바꿔 보려면 "beomi/KcELECTRA-small-v2022" (LR 3e-4, EPOCHS 40 권장)
LR = 5e-5
BATCH_SIZE = 16
EPOCHS = 20
PATIENCE = 6
VAL_RATIO = 0.15
MAX_LENGTH = 64
SEED = 1
RETRAIN = True  # False 면 학습을 건너뛰고 OUT 폴더에 저장된 모델을 불러온다

DATA = [resolve_repo_path("fixtures/analyze/labeled_parsed.csv"), resolve_repo_path("fixtures/analyze/train_aug.csv")]
OUT_DIR = "models/ft/notebook-demo"
OUT = resolve_repo_path(OUT_DIR)  # git 제외 폴더. 앱이 쓰는 models/analyze-classifier 는 건드리지 않는다
LABELS = train.LABELS
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("유형:", LABELS)
print("학습 기기:", DEVICE, "· torch", torch.__version__)

유형: ['profanity', 'family_insult', 'sexual_harassment', 'blame', 'instigation']
학습 기기: mps · torch 2.15.0


## 1. 데이터

`train.py` 와 같은 함수로 읽습니다. `split=test` 행은 여기서 걸러집니다.
한 문장의 정답은 유형 5개에 대한 0/1 목록입니다(**다중 라벨** — 한 문장이 욕설이면서 비난일 수 있음).

In [2]:
examples, val = train.split_validation(train.load_examples(DATA), VAL_RATIO, SEED)
print(f"학습 {len(examples)}문장 · 검증 {len(val)}문장")
print(f"{'유형':<20}{'학습':>6}{'검증':>6}")
for j, label in enumerate(LABELS):
    print(f"{label:<20}{int(sum(y[j] for _, y in examples)):>6}{int(sum(y[j] for _, y in val)):>6}")
print(f"{'(해당 없음 = 정상)':<16}{sum(not any(y) for _, y in examples):>6}{sum(not any(y) for _, y in val):>6}")
print()
for text, y in examples[:5]:
    print(y, text)

학습 436문장 · 검증 74문장
유형                      학습    검증
profanity               73    11
family_insult           31     4
sexual_harassment       18     2
blame                  175    30
instigation             31     5
(해당 없음 = 정상)       166    29

[0.0, 0.0, 0.0, 0.0, 0.0] 탑 잘 버텼어 고생했다
[0.0, 0.0, 0.0, 0.0, 0.0] 지금은 못 가 궁 없어
[0.0, 0.0, 0.0, 0.0, 0.0] 내 탓이야 미안
[0.0, 0.0, 0.0, 0.0, 0.0] 정글 미안 내가 못했네
[0.0, 0.0, 0.0, 0.0, 0.0] 억제기 나올 때까지 버티자


## 2. 배치 — 문장이 모델에 들어가는 모양

모델은 글자를 직접 읽지 못합니다. 토크나이저가 문장을 **토큰(낱말 조각) 번호**로 바꾸고,
한 배치 안에서 길이를 맞추려고 짧은 문장 뒤에 `[PAD]` 를 채웁니다.

BERT 계열 토크나이저는 보통 문장 맨 앞에 `[CLS]`, 맨 뒤에 `[SEP]` 라는 특수 토큰을 붙입니다. 지금 받는 `beomi/KcELECTRA-base` 의 토크나이저는 **붙이지 않습니다**(아래 마지막 줄에서 확인). `[CLS]` 를 붙여 학습해 봤지만 시험 수치가 나아지지 않아 그대로 씁니다.


In [3]:
tokenizer = AutoTokenizer.from_pretrained(BASE)


def tokens_of(text):
    """문장 → 토큰 조각 목록. 이 토크나이저는 글자를 바이트 단위로 다뤄 토큰 이름이 깨져 보이므로, 원문에서 해당 부분을 잘라 보여 준다."""
    enc = tokenizer(text, return_offsets_mapping=True)
    return [
        text[a:b] or tokenizer.convert_ids_to_tokens(i)
        for i, (a, b) in zip(enc["input_ids"], enc["offset_mapping"], strict=True)
    ]


sample = [t for t, _ in examples[:4]]
batch = tokenizer(sample, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
print("input_ids 모양 (문장 수, 토큰 수):", tuple(batch["input_ids"].shape))
for text, ids in zip(sample, batch["input_ids"], strict=False):
    pads = int((ids == tokenizer.pad_token_id).sum())
    print(f"\n{text}\n  →", " | ".join(tokens_of(text)), f"+ [PAD] × {pads}" if pads else "")
    print("    번호:", ids.tolist())
steps_per_epoch = (len(examples) + BATCH_SIZE - 1) // BATCH_SIZE
print(
    f"\n학습 {len(examples)}문장 ÷ 배치 {BATCH_SIZE} = epoch 당 {steps_per_epoch}번 가중치를 고친다 (최대 {EPOCHS} epoch → {steps_per_epoch * EPOCHS}번)"
)
first = tokenizer(sample[0])["input_ids"][0]
print("\n맨 앞에 [CLS] 토큰이 붙는가:", first == tokenizer.cls_token_id)

input_ids 모양 (문장 수, 토큰 수): (4, 8)

탑 잘 버텼어 고생했다
  → 탑 |  잘 |  버 | 텼 | 텼 | 어 |  고생 | 했다 
    번호: [9026, 500, 1335, 568, 123, 319, 2743, 1426]

지금은 못 가 궁 없어
  → 지금은 |  못 |  가 |  궁 |  없어 + [PAD] × 3
    번호: [3035, 491, 389, 2035, 1708, 3, 3, 3]

내 탓이야 미안
  → 내 |  탓 | 이야 |  미안 + [PAD] × 4
    번호: [666, 2514, 1210, 3885, 3, 3, 3, 3]

정글 미안 내가 못했네
  → 정 | 글 |  미안 |  내가 |  못 | 했네 + [PAD] × 2
    번호: [333, 805, 3885, 1492, 491, 3071, 3, 3]

학습 436문장 ÷ 배치 16 = epoch 당 28번 가중치를 고친다 (최대 20 epoch → 560번)

맨 앞에 [CLS] 토큰이 붙는가: False


## 3. 학습 — 학습률 · 배치 · epoch

`train.py` 의 `train()` 과 같은 방식을, 과정이 보이도록 풀어 쓴 것입니다.

- **배치마다**: 문장 16개를 넣어 점수를 내고 → 정답과의 차이(손실)를 재고 → 손실이 줄어드는 쪽으로 가중치를 `학습률`만큼 고칩니다.
- **학습률 스케줄**: 처음 10%는 0에서 `LR` 까지 올리고(워밍업), 그 뒤로는 0까지 곧게 내립니다.
- **epoch 마다**: 검증 문장으로 유형별 F1 평균을 재고, 가장 좋았던 때의 가중치를 기억합니다. `PATIENCE` 동안 안 오르면 멈춥니다.
- **불균형 보정**: 정치질·성희롱처럼 드문 유형은 양성 가중치(`pos_weight`)로 손실을 키웁니다.

In [4]:
def encode(texts, device):
    return tokenizer(texts, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt").to(device)


def predict_probs(net, texts, device):
    net.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(texts), 64):
            out += torch.sigmoid(net(**encode(texts[i : i + 64], device)).logits).cpu().tolist()
    net.train()
    return out


history = []
if RETRAIN:
    random.seed(SEED)
    torch.manual_seed(SEED)
    net = AutoModelForSequenceClassification.from_pretrained(
        BASE,
        num_labels=len(LABELS),
        problem_type="multi_label_classification",
        id2label=dict(enumerate(LABELS)),
        label2id={label: i for i, label in enumerate(LABELS)},
    ).to(DEVICE)
    positives = [sum(y[j] for _, y in examples) for j in range(len(LABELS))]
    pos_weight = torch.tensor([min(10.0, max(1.0, (len(examples) - p) / p)) if p else 1.0 for p in positives])
    print("양성 가중치:", {label: round(w, 1) for label, w in zip(LABELS, pos_weight.tolist(), strict=False)})
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(DEVICE))
    optimizer = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=0.01)
    total_steps = EPOCHS * steps_per_epoch
    scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * 0.1), total_steps)
    train_rows = list(examples)
    best = {"score": -1.0, "epoch": 0, "state": None}
    step = 0
    started = time.perf_counter()
    net.train()
    for epoch in range(1, EPOCHS + 1):
        random.shuffle(train_rows)
        total = 0.0
        for i in range(0, len(train_rows), BATCH_SIZE):
            chunk = train_rows[i : i + BATCH_SIZE]
            labels = torch.tensor([y for _, y in chunk], device=DEVICE)
            loss = loss_fn(net(**encode([t for t, _ in chunk], DEVICE)).logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            step += 1
            total += loss.item() * len(chunk)
            if epoch == 1 and (step <= 5 or step == steps_per_epoch):  # 첫 epoch 은 배치 단위로도 보여 준다
                print(
                    f"    배치 {step:>2}/{steps_per_epoch}  문장 {len(chunk):>2}개  학습률 {scheduler.get_last_lr()[0]:.2e}  손실 {loss.item():.4f}"
                )
        score, per = train.macro_f1([y for _, y in val], predict_probs(net, [t for t, _ in val], DEVICE))
        row = {
            "epoch": epoch,
            "loss": total / len(train_rows),
            "lr": scheduler.get_last_lr()[0],
            "val_f1": score,
            "per": per,
        }
        history.append(row)
        mark = ""
        if score > best["score"]:
            best = {
                "score": score,
                "epoch": epoch,
                "state": {k: v.detach().cpu().clone() for k, v in net.state_dict().items()},
            }
            mark = "  ← 지금까지 최고"
        print(
            f"epoch {epoch:>2}/{EPOCHS}  손실 {row['loss']:.4f}  학습률 {row['lr']:.2e}  검증 F1 평균 {score:.3f}{mark}"
        )
        if epoch - best["epoch"] >= PATIENCE:
            print(f"→ {PATIENCE} epoch 동안 나아지지 않아 조기 종료")
            break
    net.load_state_dict(best["state"])
    net.to("cpu").eval()
    OUT.mkdir(parents=True, exist_ok=True)
    net.save_pretrained(OUT)
    tokenizer.save_pretrained(OUT)
    meta = {
        "version": "notebook-demo",
        "labels": LABELS,
        "max_length": MAX_LENGTH,
        "base": BASE,
        "lr": LR,
        "seed": SEED,
    }
    (OUT / "meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    print(
        f"\n가장 좋은 epoch {best['epoch']} (검증 F1 평균 {best['score']:.3f}) 저장 → {OUT_DIR}  · {time.perf_counter() - started:.0f}초"
    )
else:
    net = AutoModelForSequenceClassification.from_pretrained(OUT).eval()
    print("저장된 모델을 불러옴:", OUT_DIR)

양성 가중치: {'profanity': 5.0, 'family_insult': 10.0, 'sexual_harassment': 10.0, 'blame': 1.5, 'instigation': 10.0}


    배치  1/28  문장 16개  학습률 8.93e-07  손실 0.8631
    배치  2/28  문장 16개  학습률 1.79e-06  손실 1.1508


    배치  3/28  문장 16개  학습률 2.68e-06  손실 0.8579
    배치  4/28  문장 16개  학습률 3.57e-06  손실 1.0163


    배치  5/28  문장 16개  학습률 4.46e-06  손실 1.1091


    배치 28/28  문장  4개  학습률 2.50e-05  손실 1.3239
epoch  1/20  손실 1.0370  학습률 2.50e-05  검증 F1 평균 0.230  ← 지금까지 최고


epoch  2/20  손실 0.9993  학습률 5.00e-05  검증 F1 평균 0.429  ← 지금까지 최고


... (14 outputs left out)

epoch 17/20  손실 0.0345  학습률 8.33e-06  검증 F1 평균 0.886


epoch 18/20  손실 0.0328  학습률 5.56e-06  검증 F1 평균 0.958  ← 지금까지 최고


epoch 19/20  손실 0.0304  학습률 2.78e-06  검증 F1 평균 0.958


epoch 20/20  손실 0.0305  학습률 0.00e+00  검증 F1 평균 0.958

가장 좋은 epoch 18 (검증 F1 평균 0.958) 저장 → models/ft/notebook-demo  · 66초


### epoch 별 흐름

손실은 계속 줄지만(학습 문장을 외워 감), 검증 점수는 어느 순간부터 오르지 않습니다. 그래서 마지막이 아니라 **검증 점수가 가장 좋았던 epoch** 의 가중치를 씁니다.

In [5]:
if history:
    top = max(h["loss"] for h in history)
    print(f"{'epoch':>5}  {'학습률':>9}  {'손실':>7}  {'':<24}  {'검증F1':>6}")
    for h in history:
        print(
            f"{h['epoch']:>5}  {h['lr']:>10.2e}  {h['loss']:>8.4f}  {'█' * round(24 * h['loss'] / top):<24}  {h['val_f1']:>7.3f}  {'●' * round(20 * h['val_f1'])}"
        )
    best_row = max(history, key=lambda h: h["val_f1"])
    print("\n가장 좋은 epoch 의 유형별 검증 F1:", {k: round(v, 2) for k, v in best_row["per"].items()})

epoch        학습률       손실                              검증F1
    1    2.50e-05    1.0370  ████████████████████████    0.230  ●●●●●
    2    5.00e-05    0.9993  ███████████████████████     0.429  ●●●●●●●●●
    3    4.72e-05    0.8324  ███████████████████         0.604  ●●●●●●●●●●●●
    4    4.44e-05    0.5697  █████████████               0.707  ●●●●●●●●●●●●●●
    5    4.17e-05    0.3599  ████████                    0.787  ●●●●●●●●●●●●●●●●
    6    3.89e-05    0.2455  ██████                      0.864  ●●●●●●●●●●●●●●●●●
    7    3.61e-05    0.1834  ████                        0.780  ●●●●●●●●●●●●●●●●
    8    3.33e-05    0.1401  ███                         0.846  ●●●●●●●●●●●●●●●●●
    9    3.06e-05    0.1077  ██                          0.935  ●●●●●●●●●●●●●●●●●●●
   10    2.78e-05    0.0985  ██                          0.901  ●●●●●●●●●●●●●●●●●●
   11    2.50e-05    0.0735  ██                          0.866  ●●●●●●●●●●●●●●●●●
   12    2.22e-05    0.0613  █                           0.912  ●

## 4. 인코더 → 벡터 → 분류

네, **인코더 모델**입니다. 문장을 넣으면 글이 나오는 것이 아니라 **토큰마다 숫자 768개짜리 벡터**가 나옵니다(KcELECTRA-base 기준).
분류는 그 뒤에 붙인 작은 **분류 층**이 합니다. 파인튜닝은 인코더와 분류 층을 함께 고칩니다.

```
문장 ─토크나이저→ 토큰 번호 N개 ─인코더(12층)→ 벡터 N × 768
                                                  │ 맨 앞 위치의 벡터 1개만 꺼냄 = 문장 벡터 (768)
                                                  ▼
                                   분류 층: dense(768→768) → GELU → out_proj(768→5)
                                                  ▼
                                   유형 5개의 로짓 ─시그모이드→ 점수 0~1 ─0.5 이상→ 해당 유형
```
- **맨 앞 위치**: BERT 계열은 보통 여기에 `[CLS]` 토큰이 옵니다. 이 모델은 `[CLS]` 를 붙이지 않으므로 문장의 첫 조각 자리입니다. 인코더의 모든 위치는 문장 전체를 보고(셀프 어텐션) 벡터를 만들기 때문에, 파인튜닝하면 이 자리가 문장 전체를 요약하도록 학습됩니다.
- 시그모이드를 유형마다 **따로** 씁니다(소프트맥스 아님). 그래서 점수 합이 1이 아니고, 여러 유형이 동시에 걸릴 수 있습니다.
- `out_proj` 의 가중치는 5 × 768 행렬입니다. 한 줄 한 줄이 "이 유형다운 방향"이고, 문장 벡터와의 **내적**이 로짓입니다. 즉 분류는 대표 문장과 비교하는 것이 아니라 **학습된 방향 5개와 비교**하는 것입니다(6장에서 대표 문장 방식과 견줍니다).

In [6]:
head = net.classifier
print(
    "인코더:", type(net.electra).__name__, "· 층", net.config.num_hidden_layers, "· 벡터 크기", net.config.hidden_size
)
print("분류 층:", head)
print(
    "전체 파라미터 수:",
    f"{sum(p.numel() for p in net.parameters()):,}",
    "· 그중 분류 층:",
    f"{sum(p.numel() for p in head.parameters()):,}",
)

인코더: ElectraModel · 층 12 · 벡터 크기 768
분류 층: ElectraClassificationHead(
  (dense): Linear(in_features=768, out_features=768, bias=True)
  (activation): GELUActivation()
  (dropout): Dropout(p=0.1, inplace=False)
  (out_proj): Linear(in_features=768, out_features=5, bias=True)
)
전체 파라미터 수: 109,085,189 · 그중 분류 층: 594,437


In [7]:
text = "미드 로밍 한 번을 안 가네"
batch = encode([text], "cpu")
with torch.no_grad():
    hidden = net.electra(**batch).last_hidden_state  # (1, 토큰 수, 768)
    first = hidden[:, 0, :]  # 맨 앞 위치의 벡터 = 문장 벡터
    feature = head.activation(head.dense(first))  # 분류 층 안쪽 (768)
    logits = head.out_proj(feature)  # (1, 5)
    probs = torch.sigmoid(logits)[0]
    direct = torch.sigmoid(net(**batch).logits)[0]  # 한 번에 부른 결과와 같아야 한다

print("문장:", text)
print("토큰:", tokens_of(text))
print("인코더 출력 모양:", tuple(hidden.shape), "= (문장 1, 토큰 수, 벡터 크기)")
print("문장 벡터(맨 앞 위치) 앞 8칸:", [round(v, 3) for v in first[0, :8].tolist()], "… (총 768칸)")
print("out_proj 가중치 모양:", tuple(head.out_proj.weight.shape))
print()
print(f"{'유형':<20}{'로짓':>8}{'점수':>8}  판정(0.5 이상)")
for label, z, p in zip(LABELS, logits[0].tolist(), probs.tolist(), strict=False):
    print(f"{label:<20}{z:>8.2f}{p:>8.3f}  {'✔' if p >= 0.5 else ''}")
assert torch.allclose(probs, direct, atol=1e-5)

문장: 미드 로밍 한 번을 안 가네
토큰: ['미', '드', ' 로', '밍', ' 한', ' 번', '을', ' 안', ' 가네']
인코더 출력 모양: (1, 9, 768) = (문장 1, 토큰 수, 벡터 크기)
문장 벡터(맨 앞 위치) 앞 8칸: [-0.01, 0.766, 0.012, 0.009, -0.141, -0.011, 0.665, -0.004] … (총 768칸)
out_proj 가중치 모양: (5, 768)

유형                        로짓      점수  판정(0.5 이상)
profanity              -5.11   0.006  
family_insult          -5.71   0.003  
sexual_harassment      -5.74   0.003  
blame                   4.88   0.992  ✔
instigation            -5.11   0.006  


### 여러 문장의 점수

정상 콜, 욕설 없는 비난, 정치질, 패드립, 성희롱, 그리고 헷갈리는 문장(자기 탓, 칭찬)입니다.

In [8]:
QUERIES = [
    "바론 전에 시야 잡고 모이자",
    "정글 덕분에 편하게 이겼다",
    "내가 판단 잘못했네 미안",
    "미드 로밍 한 번을 안 가네",
    "탑 그 실력으로 랭크를 왜 돌리냐",
    "원딜 한타마다 먼저 죽네 뒤에서 딜 좀 해",
    "다들 서폿 신고 넣어라 쟤가 범인이다",
    "정글 너네 부모는 너 이러는 거 아냐",
    "서폿 여자냐 사진 좀 보내봐",
    "아 시발 핑 또 튀네",
]


def scores_of(texts):
    with torch.no_grad():
        return torch.sigmoid(net(**encode(texts, "cpu")).logits).tolist()


print("".join(f"{label[:9]:>10}" for label in LABELS), " 문장")
for text, row in zip(QUERIES, scores_of(QUERIES), strict=False):
    print("".join(f"{'*' if v >= 0.5 else ' '}{v:>9.2f}" for v in row), " ", text)
print("\n* = 0.5 이상")

 profanity family_in sexual_ha     blame instigati  문장
      0.00      0.01      0.00      0.01      0.00   바론 전에 시야 잡고 모이자
      0.01      0.00      0.00      0.01      0.00   정글 덕분에 편하게 이겼다
      0.01      0.01      0.00      0.01      0.00   내가 판단 잘못했네 미안
      0.01      0.00      0.00*     0.99      0.01   미드 로밍 한 번을 안 가네
      0.00      0.00      0.00*     0.99      0.01   탑 그 실력으로 랭크를 왜 돌리냐
      0.01      0.00      0.00*     1.00      0.01   원딜 한타마다 먼저 죽네 뒤에서 딜 좀 해
      0.04      0.02      0.02*     0.71*     0.99   다들 서폿 신고 넣어라 쟤가 범인이다
      0.03*     0.99      0.08      0.03      0.02   정글 너네 부모는 너 이러는 거 아냐
      0.10      0.20*     0.99      0.10      0.05   서폿 여자냐 사진 좀 보내봐
*     0.99      0.02      0.02      0.03      0.01   아 시발 핑 또 튀네

* = 0.5 이상


## 5. 앱에서는 이 점수를 어떻게 쓰나 (`run()`)

모델은 **유형 점수만** 냅니다. 등급·대상·가릴 조각은 `impl.py` 의 규칙이 정합니다.

- 유형마다 규칙 점수(사전에 걸리면 0.9)와 모델 점수 중 **큰 값**을 씁니다.
- **욕설(profanity)은 모델 점수를 쓰지 않습니다.** 위 표에서 모델이 욕설을 맞혀도 실제 판정은 사전이 합니다.
- 비난·정치질은 **대상이 있어야** L2 가 됩니다. 대상(포지션 낱말·닉네임)이 없으면 점수가 높아도 L0 입니다.
- 패드립·성희롱은 L3 입니다.

In [9]:
os.environ["CLEAN_CHAT_ANALYZE_MODEL"] = str(OUT)  # 이 노트북에서 학습한 모델을 run() 이 쓰게 한다
model._load.cache_clear()
for i, text in enumerate(QUERIES):
    out = impl.run(evaluate.to_request({"text": text}, i))
    types = ",".join(t.value for t in out.harm_types) or "-"
    print(f"L{int(out.level)}  대상 {out.target or '-':<4}  유형 {types:<28}  {text}")
print("\nmodel_version:", out.model_version)

L0  대상 -     유형 -                             바론 전에 시야 잡고 모이자
L0  대상 -     유형 -                             정글 덕분에 편하게 이겼다
L0  대상 -     유형 -                             내가 판단 잘못했네 미안
L2  대상 미드    유형 blame                         미드 로밍 한 번을 안 가네
L2  대상 탑     유형 blame                         탑 그 실력으로 랭크를 왜 돌리냐
L2  대상 원딜    유형 blame                         원딜 한타마다 먼저 죽네 뒤에서 딜 좀 해
L2  대상 서폿    유형 blame,instigation             다들 서폿 신고 넣어라 쟤가 범인이다
L3  대상 정글    유형 family_insult                 정글 너네 부모는 너 이러는 거 아냐
L3  대상 서폿    유형 sexual_harassment             서폿 여자냐 사진 좀 보내봐
L1  대상 -     유형 profanity                     아 시발 핑 또 튀네

model_version: rule-0.2+notebook-demo


## 6. 벡터 유사도 — 대표 문장과 비교하기

**먼저 사실 하나:** 지금 모델은 분류할 때 대표 문장을 쓰지 않습니다(4장: 학습된 방향 5개와의 내적).
여기서는 "대표 문장과 얼마나 비슷한가"로 보면 같은 결론이 나오는지 따로 확인합니다.

- 문장 벡터 = 분류 층이 읽는 것과 같은 맨 앞 위치의 벡터(768칸)를 길이 1로 맞춘 것. 두 벡터의 **코사인 유사도** = 내적(-1 ~ 1, 1에 가까울수록 비슷).
- 유형별 **대표 문장** = 학습 문장 중 그 유형으로 라벨된 것들. `정상` 은 유형이 하나도 없는 문장.
- 유형의 **대표 벡터** = 그 문장 벡터들의 평균(중심).

In [10]:
def embed(encoder_net, texts):
    """문장들 → 맨 앞 위치의 벡터 (길이 1로 정규화). 모양 (문장 수, 768)"""
    vectors = []
    with torch.no_grad():
        for i in range(0, len(texts), 64):
            vectors.append(encoder_net.electra(**encode(texts[i : i + 64], "cpu")).last_hidden_state[:, 0, :])
    return torch.nn.functional.normalize(torch.cat(vectors), dim=-1)


GROUPS = ["정상", *LABELS]
train_texts = [t for t, _ in examples]
train_vecs = embed(net, train_texts)
members = {"정상": [i for i, (_, y) in enumerate(examples) if not any(y)]}
for j, label in enumerate(LABELS):
    members[label] = [i for i, (_, y) in enumerate(examples) if y[j]]
centroids = torch.nn.functional.normalize(torch.stack([train_vecs[members[g]].mean(0) for g in GROUPS]), dim=-1)

print("유형별 대표 문장 (중심에 가장 가까운 3개)")
for g, c in zip(GROUPS, centroids, strict=False):
    sims = train_vecs[members[g]] @ c
    top = sims.topk(min(3, len(members[g]))).indices.tolist()
    print(f"  {g:<18} {len(members[g]):>3}문장 :", " / ".join(train_texts[members[g][k]] for k in top))

유형별 대표 문장 (중심에 가장 가까운 3개)
  정상                 166문장 : 한 명 끊고 바론 치자 / 리신 갱 타이밍 좋았어 / 미드 로밍 가면 핑 찍어줄게
  profanity           73문장 : 서폿 시발 진짜 / 원딜 꺼져 그냥 / 야 미드 병신이냐
  family_insult       31문장 : 탑 엄마가 그렇게 가르치디 / 서폿 가정 교육을 못 받았네 / 정글 애미 뒤졌냐
  sexual_harassment   18문장 : 속옷 무슨 색이냐 / 서폿 나랑 한번 자자 / 미드 창녀냐
  blame              175문장 : 탑 그렇게 할 거면 하지 마 / 탑 생각 좀 해라 사이드 그만 밀고 합류해 / 원딜 답 없네 템 좀 제대로 사
  instigation         31문장 : 얘들아 탑이 범인이다 신고 꼭 해 / 상대팀도 우리 탑 신고 좀 해줘 던짐 / 솔직히 이 판 미드 빼고 다 잘했다 그치


### 문장 ↔ 유형 대표 벡터의 코사인 유사도

가장 높은 칸에 `*` 를 붙였습니다. 오른쪽은 그 문장과 **가장 비슷한 학습 문장** 3개입니다.

In [11]:
query_vecs = embed(net, QUERIES)
sims = query_vecs @ centroids.T  # (문장 수, 6)
print("".join(f"{g[:9]:>10}" for g in GROUPS), " 문장")
for text, row in zip(QUERIES, sims.tolist(), strict=False):
    best_col = max(range(len(row)), key=row.__getitem__)
    print("".join(f"{'*' if k == best_col else ' '}{v:>9.2f}" for k, v in enumerate(row)), " ", text)

print("\n가장 비슷한 학습 문장 (코사인 유사도)")
nearest = (query_vecs @ train_vecs.T).topk(3, dim=-1)
for text, values, indices in zip(QUERIES, nearest.values.tolist(), nearest.indices.tolist(), strict=False):
    print(" ", text)
    for v, k in zip(values, indices, strict=False):
        tags = ",".join(label for label, hit in zip(LABELS, examples[k][1], strict=False) if hit) or "정상"
        print(f"      {v:.3f}  [{tags}]  {train_texts[k]}")

        정상 profanity family_in sexual_ha     blame instigati  문장
*     1.00      0.38      0.29      0.14      0.42      0.28   바론 전에 시야 잡고 모이자
*     1.00      0.41      0.27      0.12      0.44      0.27   정글 덕분에 편하게 이겼다
*     1.00      0.41      0.28      0.13      0.44      0.27   내가 판단 잘못했네 미안
      0.46      0.26     -0.01     -0.16*     0.98      0.29   미드 로밍 한 번을 안 가네
      0.45      0.21     -0.00     -0.14*     0.98      0.30   탑 그 실력으로 랭크를 왜 돌리냐
      0.40      0.27     -0.03     -0.16*     0.99      0.28   원딜 한타마다 먼저 죽네 뒤에서 딜 좀 해
      0.29      0.01     -0.21     -0.33      0.40*     0.99   다들 서폿 신고 넣어라 쟤가 범인이다
      0.32     -0.12*     0.98      0.09     -0.08     -0.18   정글 너네 부모는 너 이러는 거 아냐
      0.15     -0.26      0.05*     0.99     -0.18     -0.27   서폿 여자냐 사진 좀 보내봐
      0.53*     0.96      0.04     -0.10      0.21      0.02   아 시발 핑 또 튀네

가장 비슷한 학습 문장 (코사인 유사도)
  바론 전에 시야 잡고 모이자
      0.999  [정상]  바론 버프 있을 때 밀자
      0.999  [정상]  원딜 점멸 돌면 말해줘
      0.999  [정상]  바론 못 

### 대표 문장 방식 vs 분류 층 — 검증 문장으로 비교

"가장 비슷한 학습 문장 5개가 가진 유형을 다수결로 따른다"(k-최근접 이웃)와 실제 분류 층을 같은 검증 문장으로 비교합니다.
검증 문장은 학습 때 조기 종료 기준으로 쓰였으므로 두 쪽 모두 조금 낙관적인 수치입니다(시험 문장은 쓰지 않음).

In [12]:
val_texts, val_gold = [t for t, _ in val], [y for _, y in val]
val_vecs = embed(net, val_texts)
K = 5
neighbors = (val_vecs @ train_vecs.T).topk(K, dim=-1).indices.tolist()
knn_probs = [[sum(examples[k][1][j] for k in row) / K for j in range(len(LABELS))] for row in neighbors]
head_probs = scores_of(val_texts)
knn_f1, knn_per = train.macro_f1(val_gold, knn_probs)
head_f1, head_per = train.macro_f1(val_gold, head_probs)
print(f"{'유형':<20}{'분류 층':>8}{'유사도(k=5)':>12}")
for label in LABELS:
    if label in head_per:
        print(f"{label:<20}{head_per[label]:>9.2f}{knn_per[label]:>12.2f}")
print(f"{'F1 평균':<19}{head_f1:>9.3f}{knn_f1:>12.3f}")
agree = sum([p >= 0.5 for p in a] == [p >= 0.5 for p in b] for a, b in zip(knn_probs, head_probs, strict=False))
print(f"\n두 방식의 판정이 완전히 같은 문장: {agree}/{len(val)}")

유형                      분류 층    유사도(k=5)
profanity                0.91        0.91
family_insult            1.00        1.00
sexual_harassment        1.00        1.00
blame                    0.88        0.89
instigation              1.00        1.00
F1 평균                  0.958       0.960

두 방식의 판정이 완전히 같은 문장: 73/74


### 파인튜닝 전의 벡터와 비교

같은 문장을 **파인튜닝하지 않은** 원래 모델(`BASE`)로 벡터를 내면 유형이 거의 갈리지 않습니다.
원래 모델의 맨 앞 위치 벡터는 문장 뜻을 구별하도록 학습된 적이 없어서, 어떤 문장끼리도 유사도가 높게 나옵니다.
파인튜닝이 **유형별로 벡터를 떨어뜨려 놓는 일**이라는 것을 여기서 볼 수 있습니다.

In [13]:
class Raw:  # embed() 가 .electra 를 부르므로 모양만 맞춘다
    electra = AutoModel.from_pretrained(BASE).eval()


raw_train = embed(Raw, train_texts)
raw_centroids = torch.nn.functional.normalize(torch.stack([raw_train[members[g]].mean(0) for g in GROUPS]), dim=-1)


def spread(vectors, cents):
    """같은 유형 중심과의 평균 유사도 − 다른 유형 중심과의 평균 유사도 (클수록 잘 갈림)"""
    same, other = [], []
    for gi, g in enumerate(GROUPS):
        s = vectors[members[g]] @ cents.T
        same += s[:, gi].tolist()
        other += s[:, [k for k in range(len(GROUPS)) if k != gi]].mean(1).tolist()
    return sum(same) / len(same), sum(other) / len(other)


for name, vecs, cents in (("파인튜닝 전", raw_train, raw_centroids), ("파인튜닝 후", train_vecs, centroids)):
    same, other = spread(vecs, cents)
    print(f"{name}: 같은 유형 중심 {same:.3f} · 다른 유형 중심 {other:.3f} · 차이 {same - other:.3f}")

print("\n유형 중심끼리의 코사인 유사도 (파인튜닝 전 → 후)")
before, after = raw_centroids @ raw_centroids.T, centroids @ centroids.T
print(" " * 18 + "".join(f"{g[:8]:>16}" for g in GROUPS))
for gi, g in enumerate(GROUPS):
    print(f"{g[:16]:<18}" + "".join(f"{before[gi, k]:>8.2f} →{after[gi, k]:>5.2f}" for k in range(len(GROUPS))))

파인튜닝 전: 같은 유형 중심 0.947 · 다른 유형 중심 0.942 · 차이 0.004
파인튜닝 후: 같은 유형 중심 0.914 · 다른 유형 중심 0.168 · 차이 0.745

유형 중심끼리의 코사인 유사도 (파인튜닝 전 → 후)
                                정상        profanit        family_i        sexual_h           blame        instigat
정상                    1.00 → 1.00    1.00 → 0.41    0.99 → 0.28    0.99 → 0.14    1.00 → 0.42    0.99 → 0.26
profanity             1.00 → 0.41    1.00 → 1.00    1.00 →-0.01    1.00 →-0.16    1.00 → 0.34    0.99 → 0.04
family_insult         0.99 → 0.28    1.00 →-0.01    1.00 → 1.00    1.00 → 0.05    1.00 →-0.07    0.99 →-0.22
sexual_harassmen      0.99 → 0.14    1.00 →-0.16    1.00 → 0.05    1.00 → 1.00    1.00 →-0.22    0.99 →-0.33
blame                 1.00 → 0.42    1.00 → 0.34    1.00 →-0.07    1.00 →-0.22    1.00 → 1.00    1.00 → 0.39
instigation           0.99 → 0.26    0.99 → 0.04    0.99 →-0.22    0.99 →-0.33    1.00 → 0.39    1.00 → 1.00


## 정리

- 이 모델은 **인코더**라서 출력이 벡터입니다. 분류는 맨 앞 위치의 문장 벡터에 붙인 분류 층(768 → 5)이 내는 로짓에 시그모이드를 씌워 유형별 점수로 만듭니다.
- 분류 층은 대표 문장이 아니라 **학습된 방향 5개**와 비교합니다. 다만 파인튜닝 뒤에는 같은 유형의 문장 벡터가 모이므로, 대표 문장과의 코사인 유사도로 봐도 비슷한 결론이 나옵니다(6장).
- 유사도 방식의 쓸모: **왜 이렇게 판정했는지**(가장 비슷한 학습 문장)를 보여 줄 수 있고, 새 표현을 대표 문장에 추가하는 것만으로 반영할 수 있습니다. 대신 문장마다 학습 문장 전체와 비교해야 합니다(지금 규모에서는 무시할 만함).
- 한계: 문장이 모두 AI 가 쓴 가상 문장이고 검증 문장이 적습니다. 여기 수치는 흐름을 보는 용도이고, 모델 비교 수치는 README "모델 비교와 추천" 을 보세요.